In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

учёные взяли неизвестную молекулу, разбили её на фрагменты с помощью масс-спектрометра и получили список чисел.

Наша задача — по этим числам угадать, как выглядит исходная молекула.

In [ ]:
import os
import sys
import importlib.util
import subprocess

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print("Python:", sys.version)

# Проверка GPU через PyTorch
if importlib.util.find_spec("torch"):
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PyTorch: not installed")

# Проверка библиотек
print("\nLIBRARIES")
for package in [
    "pandas", "pyarrow", "numpy", "scipy",
    "rdkit", "lightgbm", "sklearn"
]:
    print(f"{package}:",
          "OK" if importlib.util.find_spec(package) else "MISSING")

# Поиск входных файлов Kaggle
print("\nINPUT FILES")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".parquet", ".csv", ".smi", ".sdf")):
            print(os.path.join(root, f))

print("\nOUTPUT DIRECTORY EXISTS:",
      os.path.isdir("/kaggle/working"))

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/input")

TRAIN_PATH = find_file("train.parquet")
TEST_PATH = find_file("test.parquet")
SAMPLE_PATH = find_file("sample_submission.csv")

print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample path:", SAMPLE_PATH)

train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest preview:")
display(test.head())

print("\nSample submission:")
display(sample.head())



train: 2 539 608 строк - Более 2,5 млн известных спектров для обучения и поиска

test: 1 213 строк - Спектры неизвестных молекул, которые нужно определить

sample: 400 строк - Пример файла, который мы должны сдать

400 molecules - Всего 400 неизвестных молекул, каждая может иметь несколько спектров





1 213 спектров, но 400 молекул,  так как одну молекулу измеряют несколько раз, например при разных энергиях столкновения. Мы должны объединить эти измерения и выдать один список предсказаний на молекулу.

In [ ]:
from collections import Counter

print("TEST adducts:")
print(test["adduct"].value_counts(dropna=False))


print("\nTRAIN adducts:")
print(train["adduct"].value_counts(dropna=False).head(20))

print("\nTest instrument types:")
print(test["instrument_type"].value_counts(dropna=False))

print("\nTest molecules:", test["molecule_id"].nunique())
print("Spectra per molecule:")
print(test.groupby("molecule_id").size().describe())

print("\nPrecursor m/z range:")
print(test["precursor_mz"].describe())

Шаг 1. Получаем спектр

Например, набор пиков с массами и интенсивностями. Это своего рода химический отпечаток молекулы.

In [ ]:
ADDUCT_DELTA = {
    "[M+H]+": 1.0072764666,
    "[M+NH4]+": 18.033823,
    "[M-H2O+H]+": -17.003289,
    "[M-2H2O+H]+": -35.013854,
    "[M+Na]+": 22.989218,
    "[M+K]+": 38.963158,
    "[M-H]-": -1.0072764666,
    "[M+CH2O2-H]-": 44.998203,
    "[M+Cl]-": 34.969402,
}

def neutral_mass(mz, adduct):
    delta = ADDUCT_DELTA.get(adduct)
    if delta is None or pd.isna(mz):
        return np.nan
    return float(mz) - delta

ref = train.copy()

ref["neutral_mass"] = [
    neutral_mass(mz, adduct)
    for mz, adduct in zip(ref["precursor_mz"], ref["adduct"])
]

ref = ref.dropna(subset=["neutral_mass", "normalized_smiles"]).copy()
ref = ref[np.isfinite(ref["neutral_mass"])].copy()
ref = ref.sort_values("neutral_mass").reset_index(drop=True)

ref_mass = ref["neutral_mass"].to_numpy(dtype=np.float64)

print("Reference spectra retained:", len(ref))
print("Unique structures:", ref["inchikey14"].nunique())
print("Mass range:", ref_mass.min(), "-", ref_mass.max())
print("Unsupported adduct spectra excluded:", len(train) - len(ref))

Что такое аддукты и почему они нас запутали?
Это самая техническая часть, но её легко понять на примере.

Масс-спектрометр измеряет не всегда массу самой молекулы, а массу заряженной формы этой молекулы.

Например:

[M+H]+ — молекула присоединила протон.

[M-H]- — молекула потеряла протон.

[M+Na]+ — молекула присоединила натрий.

[2M+H]+ — две молекулы объединились и получили протон.

Поэтому мы написали словарь ADDUCT_DELTA, который помогает пересчитать измеренную массу в приблизительную массу исходной молекулы.

Например, если прибор измерил [M+H]+ с массой 301,123, мы вычитаем массу протона и получаем примерно 300,116 Да.

Это важно, потому что иначе мы будем искать в базе молекулы с неправильной массой.

Шаг 2. Ищем похожие известные молекулы

Сравниваем неизвестный спектр с миллионами спектров, для которых структура уже известна.

проверяем, сколько кандидатов найдётся
Здесь мы пока не сравниваем спектры. Только проверяем, сколько обучающих записей имеют массу, близкую к массе неизвестной молекулы.

Допуск 20 ppm — это небольшой относительный интервал вокруг измеренной массы.

In [ ]:
MASS_TOL_PPM = 20.0

def find_mass_candidates(mz, adduct):
    mass = neutral_mass(mz, adduct)

    if not np.isfinite(mass):
        return ref.iloc[0:0]

    tol = mass * MASS_TOL_PPM * 1e-6

    left = np.searchsorted(ref_mass, mass - tol, side="left")
    right = np.searchsorted(ref_mass, mass + tol, side="right")

    return ref.iloc[left:right]

# Проверим первые пять тестовых спектров.
for idx, row in test.head(5).iterrows():
    candidates = find_mass_candidates(
        row["precursor_mz"],
        row["adduct"]
    )

    print(
        f"\nspectres {idx}: {row['molecule_id']}, "
        f"adduct={row['adduct']}, "
        f"candidates={len(candidates)}, "
        f"unique structures={candidates['inchikey14'].nunique()}"
    )

    display(
        candidates[
            ["normalized_smiles", "molecular_formula",
             "neutral_mass", "inchikey14"]
        ].head(3)
    )

**Следующий шаг — спектральное сходство**

У нас есть два источника информации:

Масса молекулы: помогает найти возможные структуры.

Фрагментные пики: помогают понять, какая из этих структур лучше объясняет экспериментальный спектр.

Например, если у неизвестной молекулы есть пики около 74.0142, 75.0238 и 76.0313, мы будем искать кандидатов, у которых соответствующие фрагменты также присутствуют.

Для начала используем косинусное сходство между спектрами. Это распространённый базовый метод сравнения масс-спектров.

In [ ]:
MASS_TOL_PPM = 20.0
FRAGMENT_TOL_DA = 0.03

def rank_molecule(molecule_id, top_n=25):
    group = test[test["molecule_id"] == molecule_id]

    if group.empty:
        raise ValueError(f"Unknown molecule_id: {molecule_id}")

    # Накопленные оценки структуры по отдельным спектрам.
    accumulated = {}

    for _, query in group.iterrows():
        qmass = neutral_mass(
            query["precursor_mz"], query["adduct"]
        )

        if not np.isfinite(qmass):
            continue

        tol = qmass * MASS_TOL_PPM * 1e-6
        left = np.searchsorted(ref_mass, qmass - tol)
        right = np.searchsorted(ref_mass, qmass + tol)

        candidates = ref.iloc[left:right]

        if candidates.empty:
            continue

        q_mz, q_int = clean_spectrum(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            query["precursor_mz"],
        )

        # Для каждой структуры сохраняем лучшее сходство
        # среди всех её эталонных спектров.
        spectrum_scores = {}

        for _, candidate in candidates.iterrows():
            key = candidate["inchikey14"]

            r_mz, r_int = clean_spectrum(
                candidate["ms2_mzs"],
                candidate["ms2_normalized_intensities"],
                candidate["precursor_mz"],
            )

            sim = spectral_cosine(
                q_mz, q_int, r_mz, r_int
            )

            if key not in spectrum_scores or sim > spectrum_scores[key][0]:
                spectrum_scores[key] = (
                    sim,
                    candidate["normalized_smiles"],
                    candidate["molecular_formula"],
                    candidate["neutral_mass"],
                )

        for key, values in spectrum_scores.items():
            sim, smiles, formula, mass = values

            if key not in accumulated:
                accumulated[key] = {
                    "smiles": smiles,
                    "formula": formula,
                    "neutral_mass": mass,
                    "scores": [],
                }

            accumulated[key]["scores"].append(sim)

    results = []

    for key, values in accumulated.items():
        scores = values["scores"]

        results.append({
            "inchikey14": key,
            "smiles": values["smiles"],
            "formula": values["formula"],
            "neutral_mass": values["neutral_mass"],
            "mean_similarity": float(np.mean(scores)),
            "best_similarity": float(np.max(scores)),
            "spectra_matched": len(scores),
        })

    result = pd.DataFrame(results)

    if result.empty:
        return result

    # Основной рейтинг — среднее сходство по спектрам,
    # для которых структура прошла фильтр массы.
    return (
        result.sort_values(
            ["mean_similarity", "best_similarity"],
            ascending=False,
        )
        .head(top_n)
        .reset_index(drop=True)
    )


ranked_all = rank_molecule("m_005e53", top_n=25)
display(ranked_all)

In [ ]:
molecule_id = "m_005e53"

print("Спектры молекулы:")
display(
    test.loc[
        test["molecule_id"] == molecule_id,
        [
            "spectrum_id",
            "adduct",
            "precursor_mz",
            "collision_energy_ev",
        ],
    ]
)

print("\nТоп кандидатов:")
display(
    ranked_all[
        [
            "smiles",
            "formula",
            "mean_similarity",
            "best_similarity",
            "spectra_matched",
        ]
    ].head(10)
)

проверяем, не является ли это дубликатом


In [ ]:
# Берём первого кандидата из твоего рейтинга.
best = ranked_all.iloc[0]
best_key = best["inchikey14"]

print("Best candidate InChIKey14:", best_key)
print("SMILES:", best["smiles"])

# Находим все обучающие записи этой структуры.
matches = ref[ref["inchikey14"] == best_key].copy()

print("\nОбучающих спектров этой структуры:", len(matches))
print("\nИсточники:")
if "ingest_lib" in matches.columns:
    display(matches["ingest_lib"].value_counts().head(15))

# Сравниваем тестовые спектры с лучшими эталонными спектрами
# именно этой структуры.
check_rows = []

for _, query in test[test["molecule_id"] == "m_005e53"].iterrows():
    q_mz, q_int = clean_spectrum(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        query["precursor_mz"],
    )

    best_score = -1
    best_ref = None

    for _, candidate in matches.iterrows():
        r_mz, r_int = clean_spectrum(
            candidate["ms2_mzs"],
            candidate["ms2_normalized_intensities"],
            candidate["precursor_mz"],
        )

        score = spectral_cosine(q_mz, q_int, r_mz, r_int)

        if score > best_score:
            best_score = score
            best_ref = candidate

    check_rows.append({
        "spectrum_id": query["spectrum_id"],
        "test_adduct": query["adduct"],
        "test_energy": str(query["collision_energy_ev"]),
        "best_similarity": best_score,
        "reference_library": (
            best_ref["ingest_lib"] if best_ref is not None else None
        ),
        "reference_adduct": (
            best_ref["adduct"] if best_ref is not None else None
        ),
        "reference_energy": (
            best_ref["collision_energy_orig"] if best_ref is not None else None
        ),
        "reference_precursor_mz": (
            best_ref["precursor_mz"] if best_ref is not None else np.nan
        ),
    })

display(pd.DataFrame(check_rows))

Cледующий код проверит, для скольких тестовых молекул в enveda-180 есть структура с совпадающей массой. Это ещё не поиск точных спектральных совпадений, но полезная оценка покрытия.

In [ ]:
import numpy as np
import pandas as pd

MASS_TOL_PPM = 20.0

# Отдельный справочник спектров с того же прибора.
ref_enveda = ref[
    ref["ingest_lib"].eq("enveda-180")
].copy()

ref_enveda = ref_enveda.sort_values("neutral_mass").reset_index(drop=True)
enveda_mass = ref_enveda["neutral_mass"].to_numpy(dtype=np.float64)

def enveda_mass_candidates(mz, adduct):
    mass = neutral_mass(mz, adduct)

    if not np.isfinite(mass):
        return 0

    tol = mass * MASS_TOL_PPM * 1e-6

    left = np.searchsorted(enveda_mass, mass - tol, side="left")
    right = np.searchsorted(enveda_mass, mass + tol, side="right")

    return right - left

test["enveda_mass_candidates"] = [
    enveda_mass_candidates(row["precursor_mz"], row["adduct"])
    for _, row in test.iterrows()
]

coverage = (
    test.groupby("molecule_id")["enveda_mass_candidates"]
    .max()
)

print("Всего молекул:", len(coverage))
print("Молекул с кандидатами в enveda-180:", (coverage > 0).sum())
print("Молекул без кандидатов:", (coverage == 0).sum())

print("\nРаспределение числа кандидатов на спектр:")
print(test["enveda_mass_candidates"].describe())

display(
    test[[
        "molecule_id", "spectrum_id", "adduct",
        "precursor_mz", "enveda_mass_candidates"
    ]].head(15)
)

итоговый алгоритм будет работать в три этапа:

Точное спектральное совпадение: искать соответствия в enveda-180 и других библиотеках.

Спектральное сходство: если точного совпадения нет, ранжировать структуры с подходящей массой по сходству фрагментов.

Резервные кандидаты: если библиотека не даёт хороших совпадений, использовать более широкий поиск по обучающим структурам.

Шаг 3. Предсказываем структуру

Выбираем до 25 наиболее вероятных молекул и записываем их в формате SMILES.

In [ ]:
from collections import defaultdict

# --------------------------------------------------
# 1. Подготовка справочника
# --------------------------------------------------

MASS_TOL_PPM = 20.0
FRAGMENT_TOL_DA = 0.03
TOP_N = 25

ref_enveda = ref[
    ref["ingest_lib"].eq("enveda-180")
].copy()

ref_enveda = (
    ref_enveda
    .dropna(subset=["neutral_mass", "normalized_smiles", "inchikey14"])
    .sort_values("neutral_mass")
    .reset_index(drop=True)
)

enveda_mass = ref_enveda["neutral_mass"].to_numpy(dtype=np.float64)

print("Эталонных спектров:", len(ref_enveda))
print("Уникальных структур:", ref_enveda["inchikey14"].nunique())


# --------------------------------------------------
# 2. Ранжирование кандидатов для одной молекулы
# --------------------------------------------------

def rank_molecule_final(molecule_id, top_n=TOP_N):
    group = test[test["molecule_id"] == molecule_id]

    # Словарь: структура -> оценки по спектрам
    scores_by_structure = defaultdict(list)
    smiles_by_structure = {}
    formula_by_structure = {}

    for _, query in group.iterrows():
        qmass = neutral_mass(
            query["precursor_mz"], query["adduct"]
        )

        if not np.isfinite(qmass):
            continue

        tol = qmass * MASS_TOL_PPM * 1e-6

        left = np.searchsorted(
            enveda_mass, qmass - tol, side="left"
        )
        right = np.searchsorted(
            enveda_mass, qmass + tol, side="right"
        )

        candidates = ref_enveda.iloc[left:right]

        if candidates.empty:
            continue

        # Сначала предпочитаем тот же аддукт.
        same_adduct = candidates[
            candidates["adduct"] == query["adduct"]
        ]

        if not same_adduct.empty:
            candidates = same_adduct

        q_mz, q_int = clean_spectrum(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            query["precursor_mz"],
        )

        # Для каждой структуры оставляем лучший спектральный
        # результат среди её записей в текущем пуле кандидатов.
        per_query = {}

        for _, candidate in candidates.iterrows():
            key = candidate["inchikey14"]

            r_mz, r_int = clean_spectrum(
                candidate["ms2_mzs"],
                candidate["ms2_normalized_intensities"],
                candidate["precursor_mz"],
            )

            sim = spectral_cosine(
                q_mz, q_int, r_mz, r_int,
                tol=FRAGMENT_TOL_DA,
            )

            if key not in per_query or sim > per_query[key][0]:
                per_query[key] = (
                    sim,
                    candidate["normalized_smiles"],
                    candidate["molecular_formula"],
                )

        for key, (sim, smiles, formula) in per_query.items():
            scores_by_structure[key].append(float(sim))
            smiles_by_structure[key] = smiles
            formula_by_structure[key] = formula

    # Сводим оценки нескольких спектров в один рейтинг.
    results = []

    for key, scores in scores_by_structure.items():
        results.append({
            "inchikey14": key,
            "smiles": smiles_by_structure[key],
            "formula": formula_by_structure[key],
            "mean_similarity": float(np.mean(scores)),
            "best_similarity": float(np.max(scores)),
            "n_spectra_scored": len(scores),
        })

    if not results:
        return pd.DataFrame(
            columns=[
                "inchikey14", "smiles", "formula",
                "mean_similarity", "best_similarity",
                "n_spectra_scored",
            ]
        )

    result = pd.DataFrame(results)

    return (
        result.sort_values(
            ["mean_similarity", "best_similarity"],
            ascending=False,
        )
        .drop_duplicates("inchikey14")
        .head(top_n)
        .reset_index(drop=True)
    )


# --------------------------------------------------
# 3. Обрабатываем все молекулы
# --------------------------------------------------

molecule_ids = test["molecule_id"].drop_duplicates().tolist()
rankings = {}

for i, molecule_id in enumerate(molecule_ids, start=1):
    rankings[molecule_id] = rank_molecule_final(molecule_id)

    if i % 25 == 0 or i == len(molecule_ids):
        print(f"Обработано {i}/{len(molecule_ids)} молекул")


# --------------------------------------------------
# 4. Создаём submission.csv
# --------------------------------------------------

# sample задаёт требуемый формат и порядок molecule_id.
submission_rows = []

for molecule_id in sample["molecule_id"]:
    ranked = rankings.get(molecule_id)

    if ranked is None or ranked.empty:
        # Не оставляем пустые поля: резервный поиск по всей базе.
        group = test[test["molecule_id"] == molecule_id]
        qmass = neutral_mass(
            group.iloc[0]["precursor_mz"],
            group.iloc[0]["adduct"],
        )

        tol = qmass * MASS_TOL_PPM * 1e-6
        left = np.searchsorted(ref_mass, qmass - tol)
        right = np.searchsorted(ref_mass, qmass + tol)

        fallback = (
            ref.iloc[left:right]
            .drop_duplicates("inchikey14")
            .head(TOP_N)
        )

        smiles_list = fallback["normalized_smiles"].tolist()
    else:
        smiles_list = ranked["smiles"].drop_duplicates().tolist()

    # Формат конкурса: несколько SMILES через точку с запятой.
    smiles_list = [s for s in smiles_list if isinstance(s, str) and s.strip()]
    smiles_list = smiles_list[:TOP_N]

    if not smiles_list:
        raise ValueError(
            f"Нет кандидатов для {molecule_id}; "
            "нужно проверить справочник и массы."
        )

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(smiles_list),
    })

submission = pd.DataFrame(submission_rows)

output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

# --------------------------------------------------
# 5. Проверяем формат файла
# --------------------------------------------------

assert list(submission.columns) == ["molecule_id", "smiles"]
assert len(submission) == sample["molecule_id"].nunique()
assert submission["molecule_id"].is_unique
assert submission["smiles"].notna().all()
assert submission["smiles"].str.len().gt(0).all()
assert submission["smiles"].apply(
    lambda x: len(x.split(";")) <= TOP_N
).all()

print("\nФайл создан:", output_path)
print("Количество строк:", len(submission))
print("Максимум кандидатов:", submission["smiles"].str.count(";").max() + 1)
display(submission.head())

In [37]:

path = "/kaggle/working/submission.csv"

print("Файл существует:", os.path.exists(path))
print("Содержимое /kaggle/working:")
print(os.listdir("/kaggle/working"))

if os.path.exists(path):
    df = pd.read_csv(path)
    print("\nРазмер:", df.shape)
    print("Колонки:", df.columns.tolist())
    print("Пустые значения:\n", df.isna().sum())
    display(df.head())

Файл существует: True
Содержимое /kaggle/working:
['.virtual_documents', 'submission.csv']

Размер: (400, 2)
Колонки: ['molecule_id', 'smiles']
Пустые значения:
 molecule_id    0
smiles         0
dtype: int64


,molecule_id,smiles
0,m_005e53,CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1;Cc1ccc...
1,m_006153,COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;COc1...
2,m_00b5aa,COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;COC1CN(Cc2nnn...
3,m_01e922,Cc1ccc(C(=O)OCC2CCCCC2)c(=O)[nH]1;CCC(=O)Nc1cc...
4,m_0259d4,O=C(NCCCn1ccnc1)NCCc1coc(-c2ccccc2)n1;Cc1ccc2[...


In [38]:
submission_path = "/kaggle/working/submission.csv"

assert os.path.isfile(submission_path), "submission.csv отсутствует"

print("Готовый файл:", submission_path)
print("Размер, байт:", os.path.getsize(submission_path))

Готовый файл: /kaggle/working/submission.csv
Размер, байт: 365605
